# ipyniivue smoke test

This notebook exercises the Phase 2.x ipyniivue widget end-to-end:
constructor overrides, reactive property sync, command-method dispatch,
and event subscription.

The widget renders in the browser via WebGPU. When you run this notebook
in JupyterLab, the cells below display a NiiVue view. When run
headlessly (`jupyter execute`), only the Python side is exercised — the
browser-side rendering won't happen, but the cells should still execute
without raising.


In [1]:
from ipyniivue import NIIVUE_EVENT_NAMES, NiiVue

print(f"NiiVue widget class loaded.")
print(f"{len(NIIVUE_EVENT_NAMES)} known events; sample: {sorted(NIIVUE_EVENT_NAMES)[:5]}")


NiiVue widget class loaded.
28 known events; sample: ['angleCompleted', 'annotationAdded', 'annotationChanged', 'annotationRemoved', 'azimuthElevationChange']


## Construct the widget

We pass a couple of overrides via constructor kwargs. Anything we don't
set will reflect NiiVue's actual default once the widget attaches in the
browser (the JS shim seeds Python with `nv`'s post-construction values).


In [2]:
nv = NiiVue(slice_type=4, is_colorbar_visible=True)
nv  # display


NiiVue(is_colorbar_visible=True, slice_type=4.0)

## Subscribe to a NiiVue event

We register a Python callback for `locationChange`. As the user moves
the crosshair, the JS shim sends event details to Python. The list
below collects them.


In [11]:
locations: list = []

def on_location(detail):
    locations.append(detail)
    if detail and "string" in detail:
        # Print the human-readable location string the widget shows in
        # its footer when there's a location.
        print(detail["string"])

unsubscribe = nv.on("locationChange", on_location)
print("Subscribed to locationChange.")


Subscribed to locationChange.


## Load a volume

The volume comes from the niivue-demo-images CDN (CORS-enabled GitHub
Pages mirror). `load_volumes` is a fire-and-forget command — it sends
a message to the JS side and returns immediately.

URL: `https://niivue.github.io/niivue-demo-images/mni152.nii.gz`


In [12]:
nv.load_volumes([{"url": "https://niivue.github.io/niivue-demo-images/mni152.nii.gz"}])
print("load_volumes message sent.")


load_volumes message sent.


## Toggle a reactive property

Setting `is_colorbar_visible = False` sends a state-change to the JS
side via the anywidget comm. After this cell runs, the colorbar in the
widget above should disappear (only visible when running interactively).


In [13]:
nv.is_colorbar_visible = False
print(f"is_colorbar_visible = {nv.is_colorbar_visible}")


is_colorbar_visible = False


## Read back current values

Properties we didn't explicitly set should reflect NiiVue's true
defaults (after the widget has attached in the browser). When run
headlessly, they'll be the placeholder traitlet defaults from
`_generated.py` because the JS seeding step never ran.


In [14]:
print(f"slice_type: {nv.slice_type}")
print(f"azimuth: {nv.azimuth}")
print(f"elevation: {nv.elevation}")
print(f"is_colorbar_visible: {nv.is_colorbar_visible}")


slice_type: 4.0
azimuth: None
elevation: None
is_colorbar_visible: False


## Request/response: get a value from the JS side

Methods that return values (e.g. `get_crosshair_pos`) use a correlation-
ID round-trip. Python sends a request and `await`s a Future tied to
that id; JS executes the call and posts the result back tagged with
the same id.

In headless mode there's no browser to respond, so the Future never
resolves and we hit the timeout. In a real browser session, this cell
prints the actual crosshair position.


In [15]:
import asyncio

try:
    pos = await asyncio.wait_for(nv.get_crosshair_pos(), timeout=1.0)
    print(f"crosshair_pos: {pos}")
except asyncio.TimeoutError:
    print("(headless mode: no JS to respond; expected timeout)")


(headless mode: no JS to respond; expected timeout)


## Cleanup

Unsubscribe from the event listener and report what we captured during
this session.


In [8]:
unsubscribe()
print(f"Captured {len(locations)} locationChange events.")


Captured 0 locationChange events.


In [16]:
import jupyterlab, ipywidgets, anywidget, sys
print(f"python:     {sys.version.split()[0]}")
print(f"jupyterlab: {jupyterlab.__version__}")
print(f"ipywidgets: {ipywidgets.__version__}")
print(f"anywidget:  {anywidget.__version__}")

python:     3.13.2
jupyterlab: 4.4.10
ipywidgets: 8.1.8
anywidget:  0.9.19
